# Steam Games EDA


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder

from kaggle_games.split_data import get_split_data

## Data loading


In [ ]:
data_path = Path.cwd().parent / "data" / "games.json"

df = pd.read_json(data_path, orient="index")
df.replace(r"^\s*$", np.nan, regex=True, inplace=True)

pd.set_option("display.max_columns", None)

print(f"Columns loaded: {len(df.columns)}")
print(f"Rows loaded: {len(df)}")
print(df.head())

## Train-test split


In [ ]:
X_train, X_test, y_train, y_test = get_split_data(df)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Analysis


### Missing values


In [ ]:
print(X_train.isna().sum() / len(X_train))

In [ ]:
df_train = X_train.copy()
df_train["is_hit"] = y_train

features = df_train.drop(columns=["is_hit"])
target = df_train["is_hit"]

is_missing = features.isna()
is_present = features.notna()

hits_missing = is_missing.T.dot(target)
hits_present = is_present.T.dot(target)

missing_counts = is_missing.sum()
present_counts = is_present.sum()

summary = pd.DataFrame(
    {
        "Missing_Count": missing_counts,
        "Present_Count": present_counts,
        "Hit_Rate_NaN_%": (hits_missing / missing_counts.replace(0, np.nan))
        * 100,
        "Hit_Rate_Present_%": (hits_present / present_counts.replace(0, np.nan))
        * 100,
    }
)

summary = summary[
    (summary["Missing_Count"] > 0) & (summary["Present_Count"] > 0)
].copy()

summary["Difference_pp"] = (
    summary["Hit_Rate_Present_%"] - summary["Hit_Rate_NaN_%"]
)
summary = summary.sort_values(by="Difference_pp", key=abs, ascending=False)

print(summary)

In [ ]:
print(df_train[~df_train["score_rank"].isna()])

In [ ]:
X_train = X_train.dropna(subset=["name"])
y_train = y_train.loc[X_train.index]

In [ ]:
mnar_columns = [
    "metacritic_url",
    "reviews",
    "website",
    "support_url",
    "support_email",
    "short_description",
    "detailed_description",
    "about_the_game",
    "score_rank",
]

for col in mnar_columns:
    X_train[f"has_{col}"] = X_train[col].notna().astype(int)

X_train = X_train.drop(columns=mnar_columns)

print(X_train[[f"has_{col}" for col in mnar_columns]].head())

In [ ]:
cols_to_drop = ["header_image", "notes"]

X_train = X_train.drop(columns=cols_to_drop)

In [ ]:
print(X_train.isna().sum() / len(X_train))

### Data types and distributions


In [ ]:
print(X_train.dtypes)

In [ ]:
cols_to_drop = ["name", "developers", "publishers", "screenshots", "movies"]
X_train = X_train.drop(columns=cols_to_drop)

In [ ]:
print(X_train.dtypes)

In [ ]:
X_train["release_date"] = pd.to_datetime(
    X_train["release_date"], errors="coerce"
)
X_train["release_year"] = X_train["release_date"].dt.year.fillna(0).astype(int)
X_train["release_month"] = (
    X_train["release_date"].dt.month.fillna(0).astype(int)
)

X_train["language_count"] = (
    X_train["supported_languages"].str.len().fillna(0).astype(int)
)
X_train["tag_count"] = X_train["tags"].str.len().fillna(0).astype(int)

categories_str = X_train["categories"].astype(str)
X_train["is_multiplayer"] = categories_str.str.contains(
    "Multiplayer", case=False, na=False
).astype(int)

genres_str = X_train["genres"].astype(str)
X_train["is_free_to_play"] = genres_str.str.contains(
    "Free to Play", case=False, na=False
).astype(int)

cols_to_drop = [
    "release_date",
    "supported_languages",
    "full_audio_languages",
    "packages",
    "developers",
    "publishers",
    "categories",
    "genres",
    "screenshots",
    "movies",
    "tags",
    "name",
    "estimated_owners",
]

X_train = X_train.drop(columns=cols_to_drop, errors="ignore")

print(
    X_train[
        ["language_count", "tag_count", "is_multiplayer", "is_free_to_play"]
    ].head()
)

In [ ]:
print(X_train.describe())

In [ ]:
binary_cols = [
    col
    for col in X_train.columns
    if set(X_train[col].dropna().unique()) <= {0, 1}
]

mapping = {0.0: "No", 1.0: "Yes"}

for col in binary_cols:
    X_train[col] = X_train[col].astype(float).map(mapping).astype("category")

In [ ]:
num_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

continuous_num = [
    col for col in num_cols if X_train[col].nunique(dropna=True) > 10
]
to_plot_discrete = [col for col in X_train.columns if col not in continuous_num]

for col in continuous_num:
    plt.figure(figsize=(10, 4))
    sns.histplot(X_train[col].dropna(), bins=50, kde=True, color="royalblue")

    plt.title(f"Continuous Distribution: {col}", fontweight="bold")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

for col in to_plot_discrete:
    if col not in X_train.columns:
        continue

    plot_data = (
        X_train[col].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})
    )
    counts = plot_data.value_counts().nlargest(15)

    if counts.empty:
        continue

    plt.figure(figsize=(10, 4))

    ax = sns.barplot(
        x=counts.values,
        y=counts.index,
        palette="viridis",
        hue=counts.index,
        legend=False,
    )

    plt.title(f"Frequency Count: {col}", fontweight="bold")
    plt.xlabel("Count")
    plt.ylabel(col)

    max_val = counts.values.max()
    for i, v in enumerate(counts.values):
        ax.text(v + (max_val * 0.01), i, str(v), va="center", fontweight="bold")

    plt.tight_layout()
    plt.show()

In [ ]:
print(X_train[["windows", "mac", "linux"]].value_counts())

## Correlations


In [ ]:
current_num_cols = X_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

corr_df = X_train[current_num_cols].copy()
corr_df["TARGET_is_hit"] = y_train

corr_matrix = corr_df.corr(method="spearman")

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    mask=np.abs(corr_matrix) < 0.1,
)
plt.title(
    "Spearman Correlation Heatmap (Continuous Features vs Target)",
    fontweight="bold",
    fontsize=14,
)
plt.tight_layout()
plt.show()

In [ ]:
cat_features = X_train.select_dtypes(
    exclude=["int64", "float64", "int32", "float32"]
).columns.tolist()

X_cat_prep = X_train[cat_features].copy()

X_cat_prep = X_cat_prep.astype(str).replace(
    {"nan": "Missing", "<NA>": "Missing"}
)

encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X_cat_prep)

mi_scores = mutual_info_classif(
    X_encoded, y_train, discrete_features=True, random_state=42
)

mi_series = pd.Series(mi_scores, index=cat_features).sort_values(
    ascending=False
)

plt.figure(figsize=(10, 8))

sns.barplot(
    x=mi_series.values,
    y=mi_series.index,
    palette="magma",
    hue=mi_series.index,
    legend=False,
)

plt.title(
    "Mutual Information: Categorical Features vs Target (is_hit)",
    fontweight="bold",
    fontsize=14,
)
plt.xlabel("Mutual Information Score (Entropy Reduction)", fontsize=12)
plt.ylabel("Categorical Features", fontsize=12)
plt.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTop 5 Categorical Predictors:")
print(mi_series.head(5))

In [ ]:
low_mi_cols = ["windows", "has_score_rank"]

X_train = X_train.drop(
    columns=[col for col in low_mi_cols if col in X_train.columns]
)

In [ ]:
contingency_table = pd.crosstab(
    X_train["has_support_email"],
    X_train["has_support_url"],
    margins=True,
    margins_name="Total",
)

print("\n=== Contingency Table: Support Email vs. Support URL ===")
print(contingency_table)
print("=" * 55)

test_table = pd.crosstab(
    X_train["has_support_email"], X_train["has_support_url"]
)
chi2_stat, p_value, dof, expected = chi2_contingency(test_table)

print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"P-value:              {p_value:.4e}")
print(f"Degrees of Freedom:   {dof}")

alpha = 0.05
print("\n--- Conclusion ---")
if p_value < alpha:
    print("Reject the null hypothesis.")
    print(
        "There is a statistically significant association between providing a support email and a support URL."
    )
else:
    print("Fail to reject the null hypothesis.")
    print(
        "There is no significant association; developers providing one does not strongly correlate with providing the other."
    )